The chatbots agent will have following
1. If user asks about some anime it will give response
2. Search web for anime content and info if not available in the data
3. tell most similar anime given what user wants to see

In [27]:
import ast
from pathlib import Path
import pandas as pd
import yaml
import httpx
import asyncio
import json
import pickle
import numpy as np
import faiss
import re
from sklearn.preprocessing import normalize
from typing import Optional

In [28]:
def load_feature_bins(config_path: str | Path = "../src/config/feature_bins.yaml") -> dict:
    path = Path(config_path)
    if not path.exists():
        path = Path("..") / "src" / "config" / "feature_bins.yaml"
    with path.open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def _is_na(value) -> bool:
    return value is None or (isinstance(value, float) and pd.isna(value))


def _build_range_labels(bins: list) -> list[str]:
    ranges = []
    for start, end in zip(bins[:-1], bins[1:]):
        if end == float("inf"):
            start_text = int(start) if float(start).is_integer() else start
            ranges.append(f"{start_text}+")
        else:
            start_text = int(start) if float(start).is_integer() else start
            end_text = int(end) if float(end).is_integer() else end
            ranges.append(f"{start_text}-{end_text}")
    return ranges


def _map_bin_value(value, cfg: dict) -> str:
    if _is_na(value):
        return ""
    labels = cfg.get("labels", [])
    bins = cfg.get("bins", [])
    if not labels or not bins:
        return value

    range_labels = _build_range_labels(bins)
    label_to_range = dict(zip(labels, range_labels))

    text_labels = cfg.get("text_labels")
    text_to_range = dict(zip(text_labels, range_labels)) if text_labels else {}

    if value in label_to_range:
        return label_to_range[value]
    if value in text_to_range:
        return text_to_range[value]
    return value

In [29]:
anime_path = Path("..") / "artifacts" / "data_preprocessing" / "anime.csv"
anime_df = pd.read_csv(anime_path)

Tools

In [30]:
def get_anime_info(anime_df: pd.DataFrame, anime_id: int, config: dict | None = None, config_path: str | Path = "../src/config/feature_bins.yaml") -> dict:
    if "anime_id" not in anime_df.columns:
        raise KeyError("anime_df must include an 'anime_id' column")

    if config is None:
        config = load_feature_bins(config_path)

    anime_id = int(anime_id)
    row = anime_df.loc[anime_df["anime_id"] == anime_id]
    if row.empty:
        return {"error": "Anime not found", "anime_id": anime_id}

    fields = [
        "anime_id",
        "title",
        "rating",
        "synopsis",
        "ep_bin",
        "dur_bin",
        "era",
        "source",
        "genres",
        "producers",
        "studios",
    ]
    record = row.iloc[0][fields].to_dict()

    for list_col in ["genres", "producers", "studios"]:
        value = record.get(list_col)
        if isinstance(value, str):
            try:
                parsed = ast.literal_eval(value)
            except (SyntaxError, ValueError):
                parsed = []
        elif _is_na(value):
            parsed = []
        else:
            parsed = value
        record[list_col] = parsed

    if _is_na(record.get("synopsis")):
        record["synopsis"] = ""

    record["ep_bin"] = _map_bin_value(record.get("ep_bin"), config.get("ep_bin", {}))
    record["dur_bin"] = _map_bin_value(record.get("dur_bin"), config.get("dur_bin", {}))
    record["era"] = _map_bin_value(record.get("era"), config.get("era", {}))

    return record


get_anime_info_fn = get_anime_info

In [31]:
sample_id = int(anime_df["anime_id"].iloc[0])
result = get_anime_info(anime_df, sample_id)

result

{'anime_id': 33041,
 'title': 'Bubuki Buranki: Hoshi no Kyojin',
 'rating': 'PG-13 - Teens 13 or older',
 'synopsis': 'Sequel of Bubuki Buranki .',
 'ep_bin': '1-13',
 'dur_bin': '10-30',
 'era': '2009-2019',
 'source': 'Original',
 'genres': ['Action', 'Drama', 'Mecha', 'Sci-Fi'],
 'producers': ['Dentsu',
  'AT-X',
  'Ultra Super Pictures',
  'Sony Music Communications',
  'Bushiroad',
  'Sammy',
  'Kadokawa Media (Taiwan)',
  'Tose'],
 'studios': ['SANZIGEN']}

In [32]:
import httpx
from typing import Optional
from dotenv import load_dotenv
import os

load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
TAVILY_URL = "https://api.tavily.com/search"


async def search_anime_info(
    user_query: str,
    anime_id: Optional[int] = None,
    anime_title: Optional[str] = None,
    max_results: int = 5,
    search_depth: str = "basic",
) -> dict:
    if anime_id is None and anime_title is None:
        raise ValueError("Provide at least one of `anime_id` or `anime_title`.")

    parts = []
    if anime_title:
        parts.append(anime_title)
    if anime_id:
        parts.append(f"anime ID {anime_id}")
    parts.append("anime")
    parts.append(user_query)
    search_query = " ".join(parts)

    payload = {
        "api_key": TAVILY_API_KEY,
        "query": search_query,
        "search_depth": search_depth,
        "max_results": max_results,
        "include_answer": True,
        "include_raw_content": False,
        "include_domains": [
            "myanimelist.net", "anilist.co", "anime-planet.com",
            "animenewsnetwork.com", "crunchyroll.com", "fandom.com",
        ],
    }

    async with httpx.AsyncClient(timeout=15) as client:
        response = await client.post(TAVILY_URL, json=payload)
        response.raise_for_status()
        data = response.json()

    results = [
        {
            "title":   r.get("title", ""),
            "url":     r.get("url", ""),
            "snippet": r.get("content", ""),
            "score":   round(r.get("score", 0), 4),
        }
        for r in data.get("results", [])
    ]

    return {
        "query_used": search_query,
        "answer":     data.get("answer", ""),
        "results":    results,
    }


def print_response(res: dict) -> None:
    print(f"\n🔍 Query : {res['query_used']}")
    if res["answer"]:
        print(f"🤖 Answer: {res['answer']}\n")
    for i, r in enumerate(res["results"], 1):
        print(f"  {i}. {r['title']}  (score: {r['score']})")
        print(f"     {r['url']}")
        print(f"     {r['snippet'][:150]}...")
        print()

In [33]:
# Cell 2 — run tests
r1 = await search_anime_info(
    user_query="number of episodes and release date",
    anime_title="Fullmetal Alchemist Brotherhood",
)
print_response(r1)

r2 = await search_anime_info(
    user_query="main characters",
    anime_id=33041,
)
print_response(r2)


🔍 Query : Fullmetal Alchemist Brotherhood anime number of episodes and release date
🤖 Answer: Fullmetal Alchemist: Brotherhood has 64 episodes, airing from April 5, 2009, to July 4, 2010.

  1. Fullmetal Alchemist: Brotherhood - Episodes - MyAnimeList.net  (score: 0.8679)
     https://myanimelist.net/anime/5114/Fullmetal_Alchemist__Brotherhood/episode
     # Fullmetal Alchemist: Brotherhood. | Fullmetal Alchemist: Brotherhood  Add to My List  |  |  | | --- | --- | | Status: |  | | Eps Seen: | / 64 | | Yo...

  2. Volume 13: Brotherhood - Fullmetal Alchemist Wiki - Fandom  (score: 0.8368)
     https://fma.fandom.com/wiki/Volume_13:_Brotherhood
     Volume Number. 13 ; Episodes. 49–51 ; Release Date. September 12, 2006 ; Language. English / Japanese ; Subtitles. English...

  3. Fullmetal Alchemist: Brotherhood  (score: 0.7641)
     https://titmouse-inc-on-cartoon-network.fandom.com/wiki/Fullmetal_Alchemist:_Brotherhood
     Fullmetal Alchemist: Brotherhood ; episodes. 38 ; Runtime. 24–

In [34]:
def _join_genres(genres) -> str:
    if isinstance(genres, list):
        return " ".join([str(g) for g in genres])
    return str(genres) if genres is not None else ""


def find_similar_anime(
    description: str,
    genres,
    anime_df: pd.DataFrame | None = None,
    k: int = 10,
    vectorizer_path: str | Path = "../artifacts/recommender/content_based/tfidf_vectorizer.pkl",
    index_path: str | Path = "../artifacts/recommender/content_based/anime.index",
    id_map_path: str | Path = "../artifacts/recommender/content_based/anime_id_to_idx.json",
) -> list[dict]:
    vectorizer_path = Path(vectorizer_path)
    index_path = Path(index_path)
    id_map_path = Path(id_map_path)

    if not (vectorizer_path.exists() and index_path.exists() and id_map_path.exists()):
        raise FileNotFoundError("Missing one or more content-based artifact files.")

    with vectorizer_path.open("rb") as handle:
        vectorizer = pickle.load(handle)

    with id_map_path.open("r", encoding="utf-8") as handle:
        anime_id_to_idx = {int(k): int(v) for k, v in json.load(handle).items()}

    idx_to_anime_id = {v: k for k, v in anime_id_to_idx.items()}
    index = faiss.read_index(str(index_path))

    soup = f"{description or ''} {_join_genres(genres)}".strip().lower()
    query = vectorizer.transform([soup])
    query_vec = normalize(query, norm="l2", axis=1).astype(np.float32).toarray()

    title_by_id = {}
    if anime_df is not None and "anime_id" in anime_df.columns:
        if "title" in anime_df.columns:
            title_by_id = dict(zip(anime_df["anime_id"].astype(int), anime_df["title"]))

    scores, neighbors = index.search(query_vec, int(k))
    results = []
    for score, idx in zip(scores[0], neighbors[0]):
        if idx == -1:
            continue
        anime_id = idx_to_anime_id.get(int(idx))
        if anime_id is None:
            continue
        results.append(
            {
                "anime_id": int(anime_id),
                "title": title_by_id.get(int(anime_id), ""),
                "similarity": float(score),
            },
        )

    return results


find_similar_anime_fn = find_similar_anime

In [35]:
test_desc = "A young hero discovers hidden powers and must protect the world from an ancient threat."
test_genres = ["Action", "Adventure", "Fantasy"]

similar = find_similar_anime(test_desc, test_genres, anime_df=anime_df, k=5)

similar

[{'anime_id': 18311, 'title': 'Meoteoldosa', 'similarity': 0.1834319531917572},
 {'anime_id': 40906,
  'title': 'Dragon Quest: Dai no Daibouken (2020)',
  'similarity': 0.17468038201332092},
 {'anime_id': 594,
  'title': 'Naruto: Takigakure no Shitou - Ore ga Eiyuu Dattebayo!',
  'similarity': 0.1746484637260437},
 {'anime_id': 47391,
  'title': 'Seven Knights Revolution: Eiyuu no Keishousha',
  'similarity': 0.173993781208992},
 {'anime_id': 42268,
  'title': 'Shouxi Yu Ling Shi',
  'similarity': 0.1703391969203949}]

In [36]:
_TITLE_STOPWORDS = {
    "the",
    "a",
    "an",
    "and",
    "of",
    "to",
    "in",
    "on",
    "for",
    "with",
    "season",
    "movie",
    "film",
    "ova",
    "special",
    "part",
    "episode",
    "series",
}


def _normalize_title(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split(" ") if t and t not in _TITLE_STOPWORDS]
    return " ".join(tokens)


def fuzzy_find_anime_titles(
    query: str,
    anime_df: pd.DataFrame,
    top_n: int = 5,
    min_score: int = 60,
    ambiguous_delta: int = 5,
) -> dict:
    try:
        from rapidfuzz import fuzz, process
    except Exception as exc:
        raise ImportError("rapidfuzz is required for fuzzy matching") from exc

    if "title" not in anime_df.columns:
        raise KeyError("anime_df must include a 'title' column")

    titles = anime_df["title"].fillna("").astype(str).tolist()
    normalized_titles = [_normalize_title(t) for t in titles]
    normalized_query = _normalize_title(query)

    if not normalized_query:
        return {"query": query, "matches": [], "needs_clarification": True}

    matches = process.extract(
        normalized_query,
        normalized_titles,
        scorer=fuzz.token_set_ratio,
        limit=top_n,
    )

    results = []
    for match_text, score, idx in matches:
        if score < min_score:
            continue
        row = anime_df.iloc[idx]
        results.append(
            {
                "anime_id": int(row.get("anime_id")) if "anime_id" in row else None,
                "title": row.get("title"),
                "score": int(score),
            }
        )

    needs_clarification = False
    if len(results) > 1:
        top_score = results[0]["score"]
        second_score = results[1]["score"]
        needs_clarification = abs(top_score - second_score) <= ambiguous_delta

    return {
        "query": query,
        "matches": results,
        "needs_clarification": needs_clarification,
    }

In [37]:
query = "fullmetal "
res = fuzzy_find_anime_titles(query, anime_df, top_n=5)

res

{'query': 'fullmetal ',
 'matches': [{'anime_id': 7902,
   'title': 'Fullmetal Alchemist: Brotherhood - 4-Koma Theater',
   'score': 100},
  {'anime_id': 664, 'title': 'Fullmetal Alchemist: Reflections', 'score': 100},
  {'anime_id': 121, 'title': 'Fullmetal Alchemist', 'score': 100},
  {'anime_id': 430,
   'title': 'Fullmetal Alchemist: The Conqueror of Shamballa',
   'score': 100},
  {'anime_id': 908,
   'title': 'Fullmetal Alchemist: Premium Collection',
   'score': 100}],
 'needs_clarification': True}

Agents

In [38]:
# ============================================================
# LangGraph Anime Agent — Thread-based Memory (MemorySaver)
# ============================================================

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")



# ── System Prompt ────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert anime assistant. You help users discover,
learn about, and find similar anime.

You have access to the following tools:
1. fuzzy_find_anime   – Find an anime by name in the local database.
                        Always call this FIRST when the user mentions an anime
                        by name, so you can resolve the correct anime_id.
2. get_anime_info     – Fetch detailed local info (synopsis, genres, rating,
                        episodes, era, studios …) using an anime_id.
3. search_anime_web   – Search the web for information not in the local data
                        (characters, news, airdate, sequels, etc.).
4. find_similar_anime – Recommend anime similar to a description + genre list.

Workflow
--------
• User names an anime  →  fuzzy_find_anime → get_anime_info
  → if info is missing / user wants more  → search_anime_web
• User wants recommendations  →  find_similar_anime (build description from
  what the user tells you, infer genres where reasonable)
• Keep answers friendly, concise, and well-formatted.
"""


# ── Tool Definitions ─────────────────────────────────────────

@tool("fuzzy_find_anime")
def fuzzy_find_anime(query: str) -> dict:
    """Search for anime by name in the local dataset.

    Always call this first when a user mentions an anime by name.
    Returns up to 5 matches with anime_id, title, and match score.
    If needs_clarification is True, ask the user to pick one.

    Args:
        query: The anime title or partial name to search for.
    """
    return fuzzy_find_anime_titles(query, anime_df, top_n=5)


@tool("get_anime_info")
def get_anime_info_tool(anime_id: int) -> dict:
    """Get detailed information about an anime from the local dataset.

    Returns title, synopsis, rating, episode count, duration, era, source,
    genres, producers, and studios.

    Args:
        anime_id: The integer anime ID (obtain via fuzzy_find_anime first).
    """
    return get_anime_info_fn(anime_df, anime_id)


@tool("search_anime_web")
def search_anime_web(
    user_query: str,
    anime_title: str,
    anime_id: int = None,
) -> dict:
    """Search the web for anime information not available in the local dataset.

    Use when the user asks about details not covered by get_anime_info
    (e.g. characters, voice actors, sequels, release schedule, news).

    Args:
        user_query: The specific question or topic to search for.
        anime_title: The anime title (required).
        anime_id: The anime ID (optional, improves search accuracy).
    """
    coro = search_anime_info(
        user_query=user_query,
        anime_title=anime_title,
        anime_id=anime_id,
    )
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except Exception as exc:
            raise RuntimeError(
                "search_anime_web requires nest_asyncio in notebooks. "
                "Install with `pip install nest_asyncio`."
            ) from exc
        return loop.run_until_complete(coro)
    return asyncio.run(coro)


@tool("find_similar_anime")
def find_similar_anime_tool(description: str, genres: list[str]) -> list[dict]:
    """Find anime similar to a given description and list of genres.

    Use when the user asks for recommendations or 'something like X'.
    Returns top 10 similar anime with anime_id, title, and similarity score.

    Args:
        description: A free-text description of what the user wants to watch.
        genres: A list of genre strings, e.g. ["Action", "Fantasy"].
    """
    return find_similar_anime_fn(description, genres, anime_df=anime_df, k=10)


_tools = [fuzzy_find_anime, get_anime_info_tool, search_anime_web, find_similar_anime_tool]


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools = llm.bind_tools(_tools)


# ── Graph Nodes ──────────────────────────────────────────────

def agent_node(state: MessagesState):
    """Call the LLM with the full message history + system prompt."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


tool_node = ToolNode(_tools)


# ── Graph Assembly ────────────────────────────────────────────
#
#   START → agent ──(tool call?)──► tools → agent
#                └──(no tool call)──► END

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # routes to "tools" or END
builder.add_edge("tools", "agent")

# MemorySaver gives each thread_id its own persistent conversation history
memory = MemorySaver()
graph  = builder.compile(checkpointer=memory)

print("✅ Anime agent compiled successfully!")
print("   Nodes :", list(graph.nodes.keys()))


✅ Anime agent compiled successfully!
   Nodes : ['__start__', 'agent', 'tools']


In [ ]:
# ── Usage ─────────────────────────────────────────────────────
#
# Each unique thread_id maintains its own conversation memory.
# The agent remembers what was discussed earlier in the same thread.

def chat(user_message: str, thread_id: str = "default") -> str:
    """Send a message to the anime agent and get a response.

    Args:
        user_message: What the user wants to ask or say.
        thread_id: Unique session identifier; same id = shared memory.

    Returns:
        The agent's reply as a plain string.
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_message}]},
        config=config,
    )
    return result["messages"][-1].content


# ── Quick Demo ────────────────────────────────────────────────
# Uncomment to test:

# # Turn 1 — fetch info about a specific anime


# # Turn 2 — agent remembers previous context (same thread_id)


# # Turn 3 — ask for similar recommendations

# # Different user / fresh memory



In [40]:
reply1 = chat("Tell me about Fullmetal Alchemist Brotherhood", thread_id="session-1")
print(reply1)

Fullmetal Alchemist: Brotherhood is an action, adventure, comedy, drama, fantasy, magic, military, and shounen anime produced by Aniplex, Square Enix, Mainichi Broadcasting System, and Studio Moriken, and animated by Bones. It aired between 1999 and 2009.

The story follows Edward and Alphonse Elric, two young brothers who attempt the forbidden act of human transmutation. This transgression costs Edward his left leg and Alphonse his entire body, with Edward sacrificing his right arm to bind Alphonse's soul to a suit of armor. Edward becomes a state alchemist, the Fullmetal Alchemist, and together they search for the mythical Philosopher's Stone to restore their bodies. Their quest embroils them in a nationwide conspiracy, revealing the true nature of the Philosopher's Stone and their country's dark history.

The series is rated R - 17+ for violence and profanity.


In [41]:
reply2 = chat("Who are the main characters?", thread_id="session-1")
print(reply2)

The main characters of Fullmetal Alchemist: Brotherhood are Edward Elric, Alphonse Elric, and Roy Mustang.


In [42]:
reply3 = chat("Recommend something similar to it", thread_id="session-1")
print(reply3)


If you enjoyed Fullmetal Alchemist: Brotherhood, you might also like these:

*   **Fullmetal Alchemist** (its predecessor)
*   **Fullmetal Alchemist: Reflections**
*   **Ta ga Tame no Alchemist**
*   **Fullmetal Alchemist: The Sacred Star of Milos** (a movie related to the series)


In [43]:
reply4 = chat("I want to watch a dark fantasy anime with magic", thread_id="session-2")
print(reply4)

Here are some dark fantasy anime with magic that you might enjoy:

*   **Meoteoldosawa Ttomae**
*   **Anime Tenchou x Touhou Project**
*   **Dragon Quest Retsuden: Roto no Monshou**
*   **Mattsu to Yanma to Moburi-san 2: Suigun Otakara to Nazotoki no Shimajima**
*   **Wizard Barristers: Benmashi Cecil**
*   **Mahoutsukai Nara Miso wo Kue!**
*   **Assault Lily Mini Anime**
*   **Mahou Gakuen Lunar! Aoi Ryuu no Himitsu**
*   **Magic Knight Rayearth Pilot**
*   **Chain Chronicle: Haecceitas no Hikari Part 3**
